In [43]:
import copy
from dataclasses import dataclass, field

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision.models import alexnet, AlexNet_Weights, resnet18, ResNet18_Weights
from datasets import load_dataset


def default_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


@dataclass
class Config:
    feature_dim: int = 512 #4096
    hidden: int = 256 #32
    epochs: int = 10
    refit_epochs: int = 10
    batch_size: int = 512
    image_batch_size: int = 64
    lr: float = 1e-3
    max_depth: int = 2
    test_size: float = 0.2
    seed: int = 42
    device: str = field(default_factory=default_device)


cfg = Config()
torch.manual_seed(cfg.seed)
print(cfg)


Config(feature_dim=512, hidden=256, epochs=10, refit_epochs=10, batch_size=512, image_batch_size=64, lr=0.001, max_depth=2, test_size=0.2, seed=42, device='mps')


In [32]:
dataset = load_dataset("microsoft/cats_vs_dogs")["train"]
splits = dataset.train_test_split(test_size=cfg.test_size, seed=cfg.seed)

#weights = AlexNet_Weights.DEFAULT
weights = ResNet18_Weights.DEFAULT
preprocess = weights.transforms()
encoder = resnet18(weights=weights)
assert encoder.fc.in_features == cfg.feature_dim
encoder.fc = nn.Identity()
#encoder = alexnet(weights=weights)
#assert encoder.classifier[6].in_features == cfg.feature_dim
#encoder.classifier[6] = nn.Identity()
encoder.to(cfg.device).requires_grad_(False)
encoder.eval()


def embedding_transforms(examples):
    return {
        "pixel_values": [preprocess(image.convert("RGB"))
                         for image in examples["image"]],
        "labels": examples["labels"],
    }


def collate_as_tuple(samples):
    return (
        torch.stack([sample["pixel_values"] for sample in samples]),
        torch.tensor([sample["labels"] for sample in samples], dtype=torch.long),
    )


train_images = splits["train"].with_transform(embedding_transforms)
test_images = splits["test"].with_transform(embedding_transforms)
train_image_loader = DataLoader(
    train_images, batch_size=cfg.image_batch_size, shuffle=False,
    collate_fn=collate_as_tuple,
)
test_image_loader = DataLoader(
    test_images, batch_size=cfg.image_batch_size, shuffle=False,
    collate_fn=collate_as_tuple,
)


In [33]:
@torch.no_grad()
def extract_embeddings(encoder, loader, cfg):
    encoder.eval()
    features, labels = [], []
    for images, target in loader:
        features.append(encoder(images.to(cfg.device)).cpu())
        labels.append(target.reshape(-1).cpu().long())
    Z, y = torch.cat(features), torch.cat(labels)
    assert Z.shape == (len(y), cfg.feature_dim)
    assert torch.isfinite(Z).all()
    assert ((y == 0) | (y == 1)).all()
    return TensorDataset(Z, y)


train_dataset = extract_embeddings(encoder, train_image_loader, cfg)
test_dataset = extract_embeddings(encoder, test_image_loader, cfg)
train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False)
print("Training embeddings:", train_dataset.tensors[0].shape)
print("Test embeddings:", test_dataset.tensors[0].shape)


Training embeddings: torch.Size([18728, 512])
Test embeddings: torch.Size([4682, 512])


In [34]:
class TreeNode(nn.Module):
    def __init__(self, feature_dim, hidden=128):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(feature_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1)
        )
        self.left_branch = None
        self.right_branch = None
        self.left_wgt = nn.Parameter(torch.tensor(0.0))
        self.right_wgt = nn.Parameter(torch.tensor(0.0))

    def forward(self, x):
        out = self.trunk(x).squeeze(-1)
        if self.left_branch is not None:
            left = (self.left_branch(x) >= 0).to(x.dtype)
            out = out + self.left_wgt.clamp_min(0) * left
        if self.right_branch is not None:
            right = (self.right_branch(x) >= 0).to(x.dtype)
            out = out - self.right_wgt.clamp_min(0) * right
        return out


In [ ]:
def ordered_loader(loader, cfg):
    return DataLoader(
        loader.dataset, batch_size=cfg.batch_size, shuffle=False,
        drop_last=False, collate_fn=loader.collate_fn,
    )


@torch.no_grad()
def evaluate(node, loader, cfg):
    node.eval()
    errors, count, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x = x.to(cfg.device)
        y = y.to(cfg.device, dtype=torch.float32).reshape(-1)
        logits = node(x)
        errors += int(((logits >= 0) != y.bool()).sum().item())
        count += y.numel()
        loss_sum += F.binary_cross_entropy_with_logits(
            logits, y, reduction="sum"
        ).item()
    if count == 0:
        raise ValueError("Empty loader")
    return {"errors": errors, "accuracy": 1 - errors / count,
            "loss": loss_sum / count, "count": count}


def fit_node(node, dataloader, cfg, name="", epochs=None):
    node.to(cfg.device)
    for branch in (node.left_branch, node.right_branch):
        if branch is not None:
            branch.requires_grad_(False)
            branch.eval()
    params = list(node.trunk.parameters()) + [node.left_wgt, node.right_wgt]
    optimizer = torch.optim.Adam(params, lr=cfg.lr)
    scan = ordered_loader(dataloader, cfg)
    total, pos = 0, 0
    for _, y in scan:
        total += y.numel()
        pos += int((y == 1).sum().item())
    if total == 0:
        raise ValueError("Cannot train on an empty dataset")
    pos_weight = torch.tensor(max(total-pos, 1) / max(pos, 1),
                              dtype=torch.float32, device=cfg.device)
    with torch.no_grad():
        node.left_wgt.clamp_(min=0)
        node.right_wgt.clamp_(min=0)

    initial = evaluate(node, scan, cfg)
    best_key = (initial["errors"], initial["loss"])
    best_state = copy.deepcopy(node.state_dict())
    for epoch in range(cfg.epochs if epochs is None else epochs):
        if best_key[0] == 0:
            break
        node.trunk.train()
        for x, y in dataloader:
            x = x.to(cfg.device)
            y = y.to(cfg.device, dtype=torch.float32).reshape(-1)
            optimizer.zero_grad(set_to_none=True)
            logits = node(x)
            loss = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)
            loss.backward()
            optimizer.step()
            with torch.no_grad():
                node.left_wgt.clamp_(min=0)
                node.right_wgt.clamp_(min=0)
        metrics = evaluate(node, scan, cfg)
        key = (metrics["errors"], metrics["loss"])
        if key < best_key:
            best_key = key
            best_state = copy.deepcopy(node.state_dict())
        print(f"{name} epoch {epoch+1}: training accuracy={metrics['accuracy']:.2%}, "
              f"errors={metrics['errors']}, unweighted BCE={metrics['loss']:.4f}")
    node.load_state_dict(best_state)
    return node.eval()


In [36]:
class ChildDataset(Dataset):
    def __init__(self, parent_dataset, indices, labels):
        self.parent_dataset = parent_dataset
        self.indices = indices.tolist()
        self.labels = labels.long().cpu()

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        sample = self.parent_dataset[self.indices[index]]
        if isinstance(sample, dict):
            x = sample["pixel_values"]
        else:
            x, _ = sample
        return x, self.labels[index]

In [ ]:
@torch.no_grad()
def make_child_loaders(node, parent_loader, cfg):
    scan_loader = DataLoader(
        parent_loader.dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        drop_last=False,
        collate_fn=parent_loader.collate_fn,
    )

    node.eval()

    predictions, labels = [], []
    for x, target in scan_loader:
        score = node(x.to(cfg.device)).reshape(-1)
        predictions.append((score >= 0).cpu())
        labels.append(target.reshape(-1).cpu().bool())

    pred = torch.cat(predictions)
    y = torch.cat(labels)

    false_negative = ~pred & y
    false_positive = pred & ~y

    left_mask = ~(pred & y)
    right_mask = pred | y

    def build_loader(mask, targets):
        indices = mask.nonzero(as_tuple=True)[0]

        child_dataset = ChildDataset(
            parent_dataset=parent_loader.dataset,
            indices=indices,
            labels=targets[indices],
        )

        return DataLoader(
            child_dataset,
            batch_size=cfg.batch_size,
            shuffle=True,
            drop_last=False,
        )

    left_loader = (
        build_loader(left_mask, false_negative)
        if false_negative.any()
        else None
    )

    right_loader = (
        build_loader(right_mask, false_positive)
        if false_positive.any()
        else None
    )

    return left_loader, right_loader

In [38]:
def fit_tree(dataloader, cfg, depth=0, name="root"):
    node = TreeNode(cfg.feature_dim, cfg.hidden).to(cfg.device)
    labels = torch.cat([y.reshape(-1).cpu()
                        for _, y in ordered_loader(dataloader, cfg)])
    if (labels == labels[0]).all():
        with torch.no_grad():
            for p in node.trunk.parameters():
                p.zero_()
            node.trunk[-1].bias.fill_(1 if labels[0].item() == 1 else -1)
        return node.eval()

    node = fit_node(node, dataloader, cfg, name=name)
    if depth >= cfg.max_depth:
        return node.eval()

    left_loader, right_loader = make_child_loaders(node, dataloader, cfg)
    if left_loader is None and right_loader is None:
        return node.eval()
    if left_loader is not None:
        node.left_branch = fit_tree(left_loader, cfg, depth+1, name+"-left")
        node.left_branch.requires_grad_(False)
    if right_loader is not None:
        node.right_branch = fit_tree(right_loader, cfg, depth+1, name+"-right")
        node.right_branch.requires_grad_(False)

    return fit_node(node, dataloader, cfg, name=name+" refit",
                    epochs=cfg.refit_epochs)


In [44]:
tree = fit_tree(train_loader, cfg)
train_metrics = evaluate(tree, ordered_loader(train_loader, cfg), cfg)
test_metrics = evaluate(tree, test_loader, cfg)
node_count = sum(isinstance(module, TreeNode) for module in tree.modules())
print("Training:", train_metrics)
print("Test:", test_metrics)
print("Nodes:", node_count)


root epoch 1: training accuracy=98.61%, errors=260, unweighted BCE=0.0381
root epoch 2: training accuracy=98.83%, errors=219, unweighted BCE=0.0319
root epoch 3: training accuracy=98.88%, errors=209, unweighted BCE=0.0297
root epoch 4: training accuracy=98.96%, errors=194, unweighted BCE=0.0282
root epoch 5: training accuracy=99.04%, errors=179, unweighted BCE=0.0266
root epoch 6: training accuracy=99.08%, errors=173, unweighted BCE=0.0273
root epoch 7: training accuracy=99.15%, errors=159, unweighted BCE=0.0236
root epoch 8: training accuracy=99.18%, errors=154, unweighted BCE=0.0229
root epoch 9: training accuracy=98.95%, errors=196, unweighted BCE=0.0295
root epoch 10: training accuracy=99.23%, errors=144, unweighted BCE=0.0235
root-left epoch 1: training accuracy=86.48%, errors=1288, unweighted BCE=0.3258
root-left epoch 2: training accuracy=87.07%, errors=1232, unweighted BCE=0.3352
root-left epoch 3: training accuracy=93.30%, errors=638, unweighted BCE=0.1500
root-left epoch 4: t